# LeRobot Policy Integration — End-to-End Walkthrough

This notebook demonstrates the full lifecycle of working with [LeRobot](https://github.com/huggingface/lerobot) policies through PhysicalAI:

| # | Section | What you'll learn |
|---|---------|-------------------|
| 1 | **Availability & Discovery** | Check installation, list supported policies |
| 2 | **Instantiation** | Universal wrapper, explicit wrappers (`ACT`, `Diffusion`), convenience aliases (`PI05`, `VQBeT`) |
| 3 | **From Dataset** | Eager initialization via `from_dataset` with feature extraction |
| 4 | **From Pretrained** | Load trained models from HuggingFace Hub |
| 5 | **Training** | `LeRobotDataModule` + Lightning `Trainer` |
| 6 | **Inference** | `select_action` on a trained policy |
| 7 | **Export** | `save_pretrained` to local directory (config.json + model.safetensors) |
| 8 | **Round-Trip** | Reload exported model with native LeRobot `PreTrainedPolicy.from_pretrained` |
| 9 | **Checkpoint Conversion** | Convert between Lightning `.ckpt` ↔ LeRobot safetensors |
| 10 | **Wrapper vs Native LeRobot** | Train both from identical weights and verify matching losses |

### Prerequisites

- PhysicalAI library installed (`pip install -e library/`)
- LeRobot ≥ 0.5.1 (`pip install lerobot`)
- GPU recommended (examples use `lerobot/pusht` which is lightweight)

---
## 1. Availability & Discovery

In [ ]:
from physicalai.policies import lerobot

print(f"LeRobot available: {lerobot.is_available()}")
print(f"Supported policies: {lerobot.list_available_policies()}")

---
## 2. Instantiation

PhysicalAI provides three ways to create a LeRobot policy:

| Approach | Class | When to use |
|----------|-------|-------------|
| **Universal wrapper** | `LeRobotPolicy(policy_name="act")` | Runtime policy selection, config-driven workflows |
| **Explicit wrapper** | `ACT(chunk_size=100)` | Type-safe, IDE autocompletion, policy-specific params |
| **Convenience alias** | `PI05(learning_rate=1e-4)` | Quick access to policies without explicit wrappers |

In [ ]:
from physicalai.policies.lerobot import LeRobotPolicy

# Universal wrapper — works with any LeRobot policy by name
policy_act = LeRobotPolicy(policy_name="act")
policy_diff = LeRobotPolicy(policy_name="diffusion")

print(f"Universal ACT:       policy_name={policy_act.policy_name!r}")
print(f"Universal Diffusion: policy_name={policy_diff.policy_name!r}")

In [ ]:
from physicalai.policies.lerobot import ACT, Diffusion

# Explicit wrappers — type-safe, with IDE autocompletion for policy-specific params
act = ACT(
    chunk_size=100,
    n_obs_steps=1,
    dim_model=256,
    n_heads=8,
    n_encoder_layers=4,
    n_decoder_layers=1,
    optimizer_lr=1e-5,
)

diffusion = Diffusion(
    n_action_steps=8,
    horizon=16,
    n_obs_steps=2,
    num_inference_steps=10,
    optimizer_lr=1e-4,
)

print(f"ACT:       chunk_size={act._policy_config['chunk_size']}")
print(f"Diffusion: horizon={diffusion._policy_config['horizon']}")

In [ ]:
from physicalai.policies.lerobot import PI05, VQBeT, TDMPC, SAC, PI0, PI0Fast

# Convenience aliases — thin wrappers that set policy_name for you
pi05 = PI05(learning_rate=1e-4)
vqbet = VQBeT(learning_rate=1e-4)

print(f"PI05:  policy_name={pi05.policy_name!r}")
print(f"VQBeT: policy_name={vqbet.policy_name!r}")
print(f"\nAll aliases: PI0, PI05, PI0Fast, VQBeT, TDMPC, SAC")

In [ ]:
from physicalai.policies.lerobot import get_lerobot_policy

# Factory function — useful for config-driven instantiation
config = {"policy_name": "act", "dim_model": 256, "chunk_size": 50}
policy = get_lerobot_policy(**config)

print(f"Factory: {type(policy).__name__}, policy_name={policy.policy_name!r}")

---
## 3. From Dataset (Eager Initialization)

`from_dataset` extracts input/output features and dataset statistics from a LeRobot dataset (or HuggingFace repo ID) and builds the policy immediately — no Trainer needed.

In [ ]:
# From a HuggingFace repo ID (lightweight — only fetches metadata)
act_eager = LeRobotPolicy.from_dataset("act", "lerobot/pusht")

print(f"Policy: {act_eager.policy_name}")
print(f"Config type: {type(act_eager.config).__name__}")
print(f"Input features: {list(act_eager.config.input_features.keys())}")
print(f"Output features: {list(act_eager.config.output_features.keys())}")

In [ ]:
# Explicit wrappers also support from_dataset
# Note: n_action_steps must be <= chunk_size (ACT constraint)
act_from_ds = ACT.from_dataset("lerobot/pusht", dim_model=256, chunk_size=50, n_action_steps=50)

print(f"ACT from dataset: {type(act_from_ds).__name__}")
print(f"  chunk_size in config: {act_from_ds.config.chunk_size}")

---
## 4. From Pretrained (HuggingFace Hub)

Load a trained model directly from the Hub. The policy is returned in eval mode with weights loaded.

In [ ]:
# Load a pretrained ACT model from the Hub
pretrained_act = ACT.from_pretrained("lerobot/act_aloha_sim_transfer_cube_human")

print(f"Type: {type(pretrained_act).__name__}")
print(f"Config: {type(pretrained_act.config).__name__}")
print(f"Training mode: {pretrained_act.training}")

In [ ]:
# The universal wrapper also supports from_pretrained
pretrained_universal = LeRobotPolicy.from_pretrained(
    "lerobot/act_aloha_sim_transfer_cube_human"
)

print(f"Type: {type(pretrained_universal).__name__}")
print(f"Detected policy_name: {pretrained_universal.policy_name!r}")

---
## 5. Training with Lightning Trainer

The standard workflow:
1. Create a policy (lazy — no features needed yet)
2. Create a `LeRobotDataModule`
3. Call `Trainer.fit()` — the Trainer automatically calls `policy.setup()` to extract features from the DataModule

In [ ]:
from physicalai.data.lerobot import LeRobotDataModule, get_delta_timestamps_from_policy

fps = 10
datamodule = LeRobotDataModule(
    repo_id="lerobot/pusht",
    train_batch_size=8,
    data_format="physicalai",
    delta_timestamps=get_delta_timestamps_from_policy("act", fps=fps),
    video_backend="pyav",
)

print(f"DataModule: repo_id='lerobot/pusht', batch_size={datamodule.train_batch_size}")

In [ ]:
# Inspect dataset structure
datamodule.setup("fit")
dataset = datamodule.train_dataset

sample = dataset[0]
print(f"Dataset size: {len(dataset)} samples")
print(f"Sample type:  {type(sample).__name__}")
print(f"State shape:  {sample.state.shape}")
print(f"Action shape: {sample.action.shape}")
if isinstance(sample.images, dict):
    for name, img in sample.images.items():
        print(f"Image '{name}': {img.shape}")
else:
    print(f"Images shape: {sample.images.shape}")

In [ ]:
import torch
from physicalai.train import Trainer

torch.set_float32_matmul_precision("medium")

# Lazy policy — features will be extracted from DataModule in setup()
policy = ACT(chunk_size=100, n_obs_steps=1, optimizer_lr=1e-5)

trainer = Trainer(
    max_steps=5,
    accelerator="auto",
    enable_checkpointing=False,
    logger=False,
    enable_progress_bar=True,
)

trainer.fit(model=policy, datamodule=datamodule)
print("\nTraining complete.")

---
## 6. Inference

`select_action` handles the full pipeline: format conversion → preprocessing (normalization, batch dim) → model forward → postprocessing (denormalization).

In [ ]:
from physicalai.data.observation import Observation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = policy.to(device)
policy.eval()

# Create an observation from a dataset sample
# The preprocessor automatically handles: batch dim, device transfer, normalization
sample = dataset[0]
observation = Observation(state=sample.state, images=sample.images, action=None)

with torch.no_grad():
    action = policy.select_action(observation)

print(f"Observation state:  {sample.state.shape}")
print(f"Observation images: {sample.images.shape}")
print(f"Predicted action:   {action.shape}")
print(f"Action values:      {action}")

---
## 7. Export with `save_pretrained`

Saves the policy in LeRobot-compatible format:
- `config.json` — full policy configuration
- `model.safetensors` — model weights
- Processor pipelines for normalization

The output is loadable by both `LeRobotPolicy.from_pretrained()` and native LeRobot's `PreTrainedPolicy.from_pretrained()`.

In [ ]:
import tempfile
from pathlib import Path

export_dir = Path(tempfile.mkdtemp()) / "act_pusht_exported"
policy.save_pretrained(export_dir)

print(f"Exported to: {export_dir}")
print(f"Contents: {sorted(p.name for p in export_dir.iterdir())}")

In [ ]:
import json

with open(export_dir / "config.json") as f:
    config_json = json.load(f)

print("config.json (first 10 keys):")
for i, (k, v) in enumerate(config_json.items()):
    if i >= 10:
        print("  ...")
        break
    print(f"  {k}: {v}")

---
## 8. Round-Trip to Native LeRobot

The exported directory is fully compatible with LeRobot's `PreTrainedPolicy.from_pretrained()`. This means you can:
1. Train in PhysicalAI → export → load in LeRobot
2. Train in LeRobot → load in PhysicalAI via `from_pretrained`

In [ ]:
from lerobot.policies.factory import get_policy_class
from lerobot.configs.policies import PreTrainedConfig

# Load with native LeRobot
native_config = PreTrainedConfig.from_pretrained(str(export_dir))
native_cls = get_policy_class(native_config.type)
native_policy = native_cls.from_pretrained(str(export_dir), config=native_config)

print(f"Native LeRobot type: {type(native_policy).__name__}")
print(f"Config type:         {native_config.type}")

In [ ]:
# Verify weights match between wrapper and native
wrapper_state = policy.lerobot_policy.state_dict()
native_state = native_policy.state_dict()

print(f"Wrapper keys: {len(wrapper_state)}")
print(f"Native keys:  {len(native_state)}")

all_match = all(
    torch.equal(wrapper_state[k].cpu(), native_state[k].cpu())
    for k in wrapper_state
    if k in native_state
)
print(f"All weights identical: {all_match}")

In [ ]:
# Reload in PhysicalAI from the same exported directory
reloaded = ACT.from_pretrained(str(export_dir))
print(f"Reloaded type: {type(reloaded).__name__}")
print(f"Round-trip complete.")

---
## 9. Checkpoint Conversion (Lightning ↔ LeRobot)

Convert between Lightning `.ckpt` checkpoints and LeRobot's `config.json` + `model.safetensors` format.

| Direction | Function | Input | Output |
|-----------|----------|-------|--------|
| Lightning → LeRobot | `lightning_to_lerobot()` | `.ckpt` file | `config.json` + `model.safetensors` |
| LeRobot → Lightning | `lerobot_to_lightning()` | `config.json` + `model.safetensors` | `.ckpt` file |

In [ ]:
# First, save a Lightning checkpoint
ckpt_dir = Path(tempfile.mkdtemp()) / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

ckpt_path = ckpt_dir / "act_demo.ckpt"
ckpt_trainer = Trainer(
    max_steps=2,
    accelerator="auto",
    enable_checkpointing=False,
    logger=False,
    enable_progress_bar=False,
)
ckpt_policy = ACT(chunk_size=100, n_obs_steps=1, optimizer_lr=1e-5)
ckpt_trainer.fit(model=ckpt_policy, datamodule=datamodule)

# Save checkpoint manually
ckpt_trainer.save_checkpoint(str(ckpt_path))
print(f"Saved checkpoint: {ckpt_path}")
print(f"Checkpoint size: {ckpt_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
from physicalai.policies.lerobot import lightning_to_lerobot, lerobot_to_lightning

# Lightning .ckpt → LeRobot (config.json + model.safetensors)
lerobot_dir = Path(tempfile.mkdtemp()) / "lerobot_converted"
lightning_to_lerobot(ckpt_path, lerobot_dir)

print(f"Converted to: {lerobot_dir}")
print(f"Contents: {sorted(p.name for p in lerobot_dir.iterdir())}")

In [ ]:
# LeRobot (config.json + model.safetensors) → Lightning .ckpt
restored_ckpt = Path(tempfile.mkdtemp()) / "restored.ckpt"
lerobot_to_lightning(lerobot_dir, restored_ckpt)

# Load the restored checkpoint
restored_policy = LeRobotPolicy.load_from_checkpoint(str(restored_ckpt))

print(f"Restored from: {restored_ckpt}")
print(f"Policy type: {type(restored_policy).__name__}")
print(f"Policy name: {restored_policy.policy_name}")

In [ ]:
# Verify checkpoint round-trip: original ckpt → lerobot → ckpt → weights match
original_state = ckpt_policy.lerobot_policy.state_dict()
restored_state = restored_policy.lerobot_policy.state_dict()

all_match = all(
    torch.allclose(original_state[k].cpu().float(), restored_state[k].cpu().float(), atol=1e-6)
    for k in original_state
    if k in restored_state
)
print(f"Checkpoint round-trip weights match: {all_match}")

---
## 10. Wrapper vs Native LeRobot Comparison

This section verifies that PhysicalAI's wrapper produces **identical training dynamics** to native LeRobot. We:
1. Create wrapper and native policies with identical weights
2. Feed the same batch through both
3. Compare the output losses

In [ ]:
from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.policies.factory import make_policy_config, get_policy_class, make_pre_post_processors

# Create wrapper policy from dataset
wrapper_policy = ACT.from_dataset("lerobot/pusht", chunk_size=100, optimizer_lr=1e-5)
wrapper_policy = wrapper_policy.to(device)
wrapper_policy.train()

# Create native LeRobot policy with identical config
native_config = wrapper_policy.config
native_cls = get_policy_class("act")
native_policy = native_cls(native_config)
native_policy = native_policy.to(device)
native_policy.train()

# Copy weights from wrapper to native so they start identical
native_policy.load_state_dict(wrapper_policy.lerobot_policy.state_dict())

print("Both policies initialized with identical weights.")
print(f"  Wrapper: {type(wrapper_policy).__name__}")
print(f"  Native:  {type(native_policy).__name__}")

In [ ]:
from lerobot.policies.factory import make_pre_post_processors

# Create a datamodule in lerobot dict format
lerobot_datamodule = LeRobotDataModule(
    repo_id="lerobot/pusht",
    train_batch_size=4,
    data_format="lerobot",
    delta_timestamps=get_delta_timestamps_from_policy("act", fps=10),
    video_backend="pyav",
)
lerobot_datamodule.setup("fit")
batch_lerobot = next(iter(lerobot_datamodule.train_dataloader()))
batch_on_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch_lerobot.items()}

# Build a preprocessor from config + dataset stats
preprocessor, _ = make_pre_post_processors(
    native_config,
    dataset_stats=lerobot_datamodule.train_dataset.meta.stats,
)

# Preprocess ONCE, feed the SAME tensor to both policies
preprocessed_batch = preprocessor(batch_on_device)

# Native LeRobot forward
torch.manual_seed(0)
native_output = native_policy(preprocessed_batch)
native_loss = native_output[0] if isinstance(native_output, tuple) else native_output

# Wrapper forward (bypass wrapper preprocessing — use lerobot_policy directly)
torch.manual_seed(0)
wrapper_output = wrapper_policy.lerobot_policy(preprocessed_batch)
wrapper_loss = wrapper_output[0] if isinstance(wrapper_output, tuple) else wrapper_output

print(f"Native loss:  {native_loss.item():.6f}")
print(f"Wrapper loss: {wrapper_loss.item():.6f}")
print(f"Difference:   {abs(native_loss.item() - wrapper_loss.item()):.2e}")
print(f"Match (atol=1e-6): {torch.allclose(wrapper_loss, native_loss, atol=1e-6)}")

### Equivalence Summary

| Criterion | Tolerance | Expected |
|-----------|-----------|----------|
| Loss (single batch, same seed) | `atol=1e-6` | Exact match |
| Loss (multi-step, same seed) | `rtol=1e-5` | Match |
| Weight drift (10 steps) | `atol=1e-5` | Match |

The wrapper delegates directly to the underlying LeRobot policy — **zero computational overhead**.

---
## Cleanup

In [ ]:
import shutil

# Clean up temporary directories
for d in [export_dir, ckpt_dir, lerobot_dir, restored_ckpt.parent]:
    if d.exists():
        shutil.rmtree(d, ignore_errors=True)

print("Temporary files cleaned up.")

---
## Summary

This notebook covered the full lifecycle:

| Step | API | Description |
|------|-----|-------------|
| Discovery | `lerobot.is_available()`, `list_available_policies()` | Check what's installed |
| Instantiation | `LeRobotPolicy`, `ACT`, `PI05`, `get_lerobot_policy` | Multiple creation paths |
| From dataset | `ACT.from_dataset("lerobot/pusht")` | Eager init from dataset metadata |
| From pretrained | `ACT.from_pretrained("lerobot/act_...")` | Load trained Hub models |
| Training | `Trainer.fit(policy, datamodule)` | Lightning training loop |
| Inference | `policy.select_action(observation)` | Predict actions |
| Export | `policy.save_pretrained("./model")` | LeRobot-compatible output |
| Round-trip | `PreTrainedPolicy.from_pretrained()` | Interop with native LeRobot |
| Checkpoint conversion | `lightning_to_lerobot()`, `lerobot_to_lightning()` | Format conversion |
| Equivalence | Wrapper vs native comparison | Identical training dynamics |

### Additional Resources

- [LeRobot Documentation](https://github.com/huggingface/lerobot)
- [Benchmark Comparison Notebook](../benchmark/lerobot_benchmark_comparison.ipynb) — inference comparison with Pi0.5 on Libero
- [Integration Tests](../../tests/integration/test_lerobot_wrapper_equivalence.py) — automated equivalence validation